In [ ]:
import zipfile

def build_zip_index(zip_path):
    zip_file = zipfile.ZipFile(zip_path, 'r')
    # 构建索引：只包含文件（排除目录），并映射文件名到 ZipInfo 对象
    index = {info.filename: info for info in zip_file.infolist() if not info.is_dir()}
    return zip_file, index

# 使用示例
zip_pdb_path = "/root/private_data/luog/AbAgKer/data/SabDab.zip"
pdb_zip, pdb_index = build_zip_index(zip_pdb_path)



In [ ]:

# 三字母氨基酸缩写转单字母（标准20种 + 常见）
from Bio.Data.PDBData import protein_letters_3to1_extended 
import zipfile
import pandas as pd
from Bio.PDB import PDBParser
from io import StringIO
import numpy as np
from tqdm import tqdm  # <-- 添加 tqdm

def build_zip_index(zip_path):
    zip_file = zipfile.ZipFile(zip_path, 'r')
    index = {info.filename: info for info in zip_file.infolist() if not info.is_dir()}
    return zip_file, index

def get_sequences_from_pdb_content(pdb_content, chain_ids):
    parser = PDBParser(QUIET=True)
    with StringIO(pdb_content) as f:
        try:
            structure = parser.get_structure("temp", f)
        except Exception:
            return {}
    
    sequences = {}
    for chain in structure.get_chains():
        if chain.id not in chain_ids:
            continue
        seq = []
        for residue in chain:
            if residue.get_id()[0] != ' ':  # 跳过 HETATM
                continue
            try:
                res_name = residue.get_resname().strip()
                aa = protein_letters_3to1_extended.get(res_name, 'X')
                if aa != 'X':
                    seq.append(aa)
            except Exception:
                continue
        if seq:
            sequences[chain.id] = ''.join(seq)
    return sequences

def is_valid_chain_id(cid):
    return cid and cid.strip() not in ('NA', 'None', '')

def parse_chain_field(chain_str):
    if pd.isna(chain_str) or not isinstance(chain_str, str):
        return []
    chain_list = chain_str.strip().split(',')
    return [c.strip() for c in chain_list if is_valid_chain_id(c)]

# ----------------------------
# 主程序开始
# ----------------------------

tsv_path = "/root/private_data/luog/AbAgKer/data/sabdab_summary_all.tsv"
zip_pdb_path = "/root/private_data/luog/AbAgKer/data/SabDab.zip"

# 加载 TSV
print("Loading TSV file...")
df = pd.read_csv(tsv_path, sep='\t', dtype=str)
df = df.replace(['NA', 'None', 'nan', ''], np.nan)

# 加载 PDB ZIP
print("Building PDB ZIP index...")
pdb_zip, pdb_index = build_zip_index(zip_pdb_path)

results = []

# 使用 tqdm 包裹迭代器，显示进度条
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing PDBs", unit="file"):
    pdb = str(row['pdb']).strip().lower() if not pd.isna(row['pdb']) else ''
    if not pdb or pdb == 'none' or len(pdb) != 4:
        continue

    hchain_list = parse_chain_field(row['Hchain'])
    lchain_list = parse_chain_field(row['Lchain'])
    agchain_list = parse_chain_field(row['antigen_chain'])

    if not hchain_list and not lchain_list:
        continue

    all_chain_ids = set(hchain_list + lchain_list + agchain_list)
    if not all_chain_ids:
        continue

    # 优先使用 Chothia 编号结构
    pdb_file_name = f"all_structures/chothia/{pdb}.pdb"
    if pdb_file_name not in pdb_index:
        pdb_file_name = f"all_structures/imgt/{pdb}.pdb"
    if pdb_file_name not in pdb_index:
        pdb_file_name = f"all_structures/raw/{pdb}.pdb"
    if pdb_file_name not in pdb_index:
        continue  # 跳过

    try:
        with pdb_zip.open(pdb_file_name) as f:
            pdb_content = f.read().decode('utf-8')

        sequences = get_sequences_from_pdb_content(pdb_content, all_chain_ids)

        def format_seq(chain_list):
            items = []
            for ch in chain_list:
                seq = sequences.get(ch, '')
                if seq:
                    items.append(f"{ch}:{seq}")
            return '[' + '; '.join(items) + ']' if items else ''

        hc_seq = format_seq(hchain_list)
        lc_seq = format_seq(lchain_list)
        ag_seq = format_seq(agchain_list)

        # 提取字段并转换为 Python 原生类型
        to_none = lambda x: None if pd.isna(x) else str(x).strip()
        result = {
            'pdb': pdb,
            'heavy_chain': hc_seq,
            'light_chain': lc_seq,
            'antigen_chain': ag_seq,
            'affinity': to_none(row.get('affinity')),
            'delta_g': to_none(row.get('delta_g')),
            'affinity_method': to_none(row.get('affinity_method')),
            'temperature': to_none(row.get('temperature')),
            'pmid': to_none(row.get('pmid'))
        }

        results.append(result)

    except Exception as e:
        # 可选：记录错误（或静默跳过）
        # tqdm.write(f"❌ Error {pdb}: {e}")
        continue

# 关闭 ZIP 文件
pdb_zip.close()

print(f"🎉 Finished. Extracted {len(results)} valid entries.")

import json

# 保存为 JSON
with open("sabdab.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("💾 Results saved to sabdab.json")

In [ ]:
results

In [ ]:
import json
import random
import numpy as np
from tqdm import tqdm
import string

flag = []
result = []
seen = set()
random.seed(42)

def generate_id(length=5):
    return ''.join(random.choices(string.ascii_uppercase, k=length))

with open("/root/private_data/luog/Data_AbAg/sabdab/result/sabdab.json", "r") as f:
    data = json.load(f)
for i in tqdm(data):
    if i["antigen_chain"]!="":
        key = (i["antigen_chain"][3:-1], i["heavy_chain"][3:-1], i["light_chain"][3:-1])
        if key not in seen:
            seen.add(key)
        else:
            continue

        result.append({
        "pdb":i["pdb"].upper()+"_"+generate_id(5), 
        "seqs":"HLX",
        "mutation":"SabDab_noinfo",
        "kd":-np.log10(float(i["affinity"])) if i["affinity"]!=None else 0.0,
        "kon":0.0,
        "koff": 0.0,
        "Hold_out_type": "AB/AG",
        "X":i["antigen_chain"][3:-1],
        "P":"",
        "H":i["heavy_chain"][3:-1],
        "L":i["light_chain"][3:-1],
        })   
    
with open("/root/private_data/luog/Data_AbAg/sabdab/result/sabdab_finally_"+str(len(result))+".json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print(len(result))


100%|██████████| 19557/19557 [00:00<00:00, 27885.67it/s]


9524
